In [71]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

all_data = pd.read_csv("../../experiment_data/separability_metrics/geometric_separability_full.csv")
topo_data = pd.read_csv("../../experiment_data/balance_metrics/cross_epoch_all_ks_1e-6_corr.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
all_metrics = all_data.columns
all_metrics = all_metrics.drop(["dataset", "split", "model", "layer", "epoch", "tensor_path", "k_neighbors", "num_points", "num_classes", "train_acc", "val_acc", "knn_accuracy"])
all_metrics = list(all_metrics)

In [63]:
all_metrics

['intra_class_mean_distance',
 'inter_class_mean_distance',
 'intra_inter_distance_ratio',
 'centroid_mean_separation',
 'centroid_min_separation',
 'centroid_mean_correlation',
 'neighborhood_purity',
 'neighbourhood_purity_15',
 'knn_accuracy_15',
 'neighbourhood_purity_20',
 'knn_accuracy_20',
 'neighbourhood_purity_25',
 'knn_accuracy_25',
 'neighbourhood_purity_30',
 'knn_accuracy_30',
 'neighbourhood_purity_35',
 'knn_accuracy_35',
 'neighbourhood_purity_40',
 'knn_accuracy_40',
 'neighbourhood_purity_45',
 'knn_accuracy_45',
 'neighbourhood_purity_60',
 'knn_accuracy_60',
 'neighbourhood_purity_80',
 'knn_accuracy_80']

In [73]:
topo_data

,split,n,train_pearson,val_pearson,train_spearman,val_spearman,k
0,full_average_branching_factor,402,-0.132993,-0.181949,0.240195,0.260822,15
1,full_colless_index,402,-0.943922,-0.945172,-0.684827,-0.686721,15
2,full_average_colless_index,402,-0.706857,-0.612765,-0.766961,-0.790196,15
3,full_sackin_index,402,-0.834508,-0.899920,-0.763863,-0.741410,15
4,full_average_sackin_index,402,-0.689551,-0.624365,-0.887807,-0.904772,15
...,...,...,...,...,...,...,...
319,cifar_resnet_colless_index,201,-0.407601,-0.527797,0.253444,0.245610,80
320,cifar_resnet_average_colless_index,201,0.327923,0.249565,0.505350,0.489526,80
321,cifar_resnet_sackin_index,201,-0.773478,-0.843684,-0.869868,-0.840570,80
322,cifar_resnet_average_sackin_index,201,-0.658571,-0.557699,-0.858331,-0.829799,80


In [85]:
topo_metrics = ["average_sackin_index", "sackin_index", "leaf_count"]

In [86]:
all_corrs = pd.read_csv("../../experiment_data/separability_metrics/geometric_separability_all_corrs.csv")

splits = {}

for row in all_corrs.itertuples():
	title = str(row.split)
	
	for metric in all_metrics:
		cut = title.split(metric)
		
		if len(cut) == 2:
			split = cut[0][:-1]
			splits[metric] = splits.get(metric, {}) | {split: row[3:]}
			break 

In [87]:
for row in topo_data.itertuples():
	title = str(row.split)
	k = row.k
 
	for metric in topo_metrics:
		cut = title.split(metric)
		key = f"{metric}_k{k}"
		if len(cut) == 2:
			split = cut[0][:-1]
			splits[key] = splits.get(key, {}) | {split: row[3:-1]}
			break

In [89]:
splits

{'centroid_mean_separation': {'full': (0.3088802421422262,
   0.6918399608907537,
   0.6088283547372234,
   0.8002546524878682),
  'cifar': (-0.3464966988539318,
   -0.2412011779941937,
   -0.4477657404252296,
   -0.4889706977162006),
  'mnist': (0.4387605725940368,
   0.4551026422721692,
   0.7727537541634915,
   0.9263856321165884),
  'densenet': (0.2494759511613816,
   0.6915453067105682,
   0.5564055039462846,
   0.7986006120481826),
  'resnet': (0.4012546404579637,
   0.7919417846060306,
   0.7418476325355069,
   0.8231887317300749),
  'cifar_densenet': (-0.4850328645368265,
   -0.3817246445484298,
   -0.3797697292359427,
   -0.467104450395868),
  'cifar_resnet': (-0.2655978582579759,
   -0.1352109790947685,
   -0.3917202293580392,
   -0.3465804827609366),
  'mnist_densenet': (0.9439153800776592,
   0.91833608946427,
   0.963567696096054,
   0.930709211896508),
  'mnist_resnet': (0.8860365745200564,
   0.8610795211633219,
   0.979942680136139,
   0.9321285862732196)},
 'centroid_m

In [90]:
labels = ("t_p", "v_p", "t_s", "v_s")

In [91]:
fig = make_subplots(rows=1, cols=1)

for metric, split_corrs in splits.items():
	first = True
	for split, corr_values in split_corrs.items():
		for i, val in enumerate(corr_values):
			group = metric
			name = f"{split}-{labels[i]}"
   
			fig.add_trace(go.Scatter(x=[metric], y=[val], mode="markers", legendgroup=group, name=name, legendgrouptitle={"text": group}, showlegend=first), row=1, col=1)
			first = False

fig.show()